In [ ]:
"""
================================================================================
S1S2-Coasts: Clustering-Based Water Mask Generation
Stage 2 — Parameter Selection, Segmentation, and Dataset Export
================================================================================

Overview
--------
This module implements Stage 2 of the S1S2-Coasts pipeline. It reads the
multi-band GeoTIFF patches exported in Stage 1, applies an ensemble of
unsupervised clustering algorithms to delineate water from non-water regions,
and writes a structured dataset of matched Sentinel-1 / Sentinel-2 /
ground-truth triplets ready for deep learning training or evaluation.

The water boundary band produced in Stage 1 (rasterised from a vector
shapefile) is used here as a structural prior: it defines the spatial
segmentation (superpixels) from which spectral features are extracted for
clustering. The cluster identified as water is determined automatically by
comparing mean water-index values (e.g. MNDWI) across clusters.

--------------------------------------------------------------------------------
Pipeline Summary
--------------------------------------------------------------------------------
  1. ImageProcessor       — Iterates over all GeoTIFF patches in a directory,
                            validates the expected band count (default: 16),
                            and orchestrates the steps below for each patch.

  2. create_label_image   — Applies morphological dilation (disk radius 1) to
                            the rasterised boundary band to close gaps in the
                            line, then runs connected-component labeling on the
                            inverted result. Each connected region becomes one
                            candidate segment (superpixel) for feature extraction.

  3. create_feature_matrix — For each segment and each band in `index_band`,
                             computes three statistics: median, variance, and
                             mean over all pixels in the segment. The resulting
                             feature vector has length 3 × len(index_band).

  4. normalize_features   — Applies Min-Max scaling to bring all features into
                            the [0, 1] range before dimensionality reduction.

  5. pca_features         — Reduces the feature matrix with PCA (default:
                            5 components, capped at the minimum of n_samples
                            and n_features) to remove correlated dimensions
                            and speed up clustering.

  6. ClusteringComparison — Performs a grid search over multiple clustering
                            algorithms and hyperparameter combinations. Each
                            configuration is evaluated with three internal
                            validity metrics:
                              • Silhouette Score        (higher is better)
                              • Davies-Bouldin Index    (lower is better)
                              • Calinski-Harabasz Index (higher is better)
                            The best hyperparameter set per algorithm is
                            selected by maximising the Silhouette Score.

  7. ImageClustering      — Re-runs each algorithm with its best parameters,
                            maps segment-level cluster labels back to 2-D
                            spatial masks, and identifies the water cluster
                            as the one with the highest mean water-index
                            value. An ensemble uncertainty mask is computed
                            as the statistical mode across all per-algorithm
                            binary masks.

  8. ExportDatasetoGDrive — Crops a 512 × 512-pixel window from the centre
                            of each patch (configurable) and writes three
                            georeferenced GeoTIFF files:
                              • S2/<id>.tif  — Sentinel-2 bands 1–13 (UInt32)
                              • S1/<id>.tif  — Sentinel-1 VV + VH     (Int32)
                              • gt/<id>.tif  — Water mask              (Byte)

  9. data_information.csv — A running metadata table recording climate zone,
                            agro-ecological zone, country/region name,
                            modality availability (S1/S2/gt), and a
                            quality-check column reserved for manual review.

--------------------------------------------------------------------------------
Active Clustering Algorithms
--------------------------------------------------------------------------------
  Algorithm                   Notes
  ─────────────────────────────────────────────────────────────────────────────
  KMeans                      Lloyd and Elkan solvers, n_clusters = 2
  AgglomerativeClustering     Ward / complete / average / single linkage
  BayesianGaussianMixture     Full / tied / diag / spherical covariance
  MeanShift                   Auto and manual bandwidth sweep
  FuzzyCMeans                 Requires scikit-fuzzy (optional, off by default)
  ─────────────────────────────────────────────────────────────────────────────
  Commented out (available):  DBSCAN, HDBSCAN, SpectralClustering, OPTICS,
                              AffinityPropagation, Birch, BisectingKMeans,
                              MiniBatchKMeans — uncomment in ClusteringComparison
                              to include in the search.

--------------------------------------------------------------------------------
Input
--------------------------------------------------------------------------------
  - Directory of multi-band GeoTIFF patches produced by Stage 1
  - Default band layout (expected_band_count = 16):
      Bands  1–13 : Sentinel-2 reflectance (B1–B12 + QA60, scaled ÷ 10 000)
      Bands 14–15 : Sentinel-1 SAR         (VV, VH)
      Band  16    : Rasterised water boundary (0 = background, 255 = boundary)
  - With spectral indices (expected_band_count = 21):
      Bands  1–13 : Sentinel-2 reflectance
      Bands 14–18 : Spectral indices (MNDWI, NDVI, NDPI, WRI, AWI)
      Bands 19–20 : Sentinel-1 SAR
      Band  21    : Rasterised water boundary

--------------------------------------------------------------------------------
Output
--------------------------------------------------------------------------------
  <root_file>/<Region Name>/
    S2/                    — Sentinel-2 crops  (UInt32, 13 bands, ×10 000 DN)
    S1/                    — Sentinel-1 crops  (Int32,  2 bands, raw)
    gt/                    — Ground-truth masks (Byte, 1 band, 0 / 255)
    data_information.csv   — Patch-level metadata

--------------------------------------------------------------------------------
Key Differences from Stage 1
--------------------------------------------------------------------------------
  Feature                Stage 1                       Stage 2
  ─────────────────────────────────────────────────────────────────────────────
  Environment            Google Earth Engine           Local / Google Colab
  Input                  Coastline / lake shapefile    Multi-band GeoTIFF patches
  Output                 Raw stacked satellite patches Structured S1 / S2 / gt
  Water mask             Rasterised vector boundary    Clustering ensemble mask
  Metadata               S1 / S2 image IDs + dates    Labels, modality flags, QC

--------------------------------------------------------------------------------
Dependencies
--------------------------------------------------------------------------------
  - scikit-learn  >= 1.3.0    (clustering algorithms, PCA, metrics)
  - scikit-image  >= 0.21.0   (morphological dilation, connected-component label)
  - GDAL          >= 3.6.0    (raster I/O, GeoTIFF creation)
  - scipy         >= 1.9.0    (statistical mode for uncertainty mask)
  - numpy         >= 1.21.0
  - pandas        >= 1.3.0
  - matplotlib    >= 3.5.0
  - seaborn       >= 0.12.0
  - tqdm          >= 4.62.0
  - scikit-fuzzy  >= 0.4.2    (optional — required only for FuzzyCMeans)

  Install: pip install -r requirements.txt

--------------------------------------------------------------------------------
Author
--------------------------------------------------------------------------------
  Soheil Zaghian
  Email  : soheil.zaghian.sz@gmail.com
  Stage  : 2 / 2
  Input  : Multi-band GeoTIFF (Stage 1 output)
================================================================================
"""



import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from osgeo import gdal, gdalconst
from skimage.measure import label
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.model_selection import ParameterGrid
from sklearn.cluster import (KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN,
                             SpectralClustering, MeanShift, OPTICS,
                             AffinityPropagation, Birch, MiniBatchKMeans,
                             BisectingKMeans)
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
import warnings
from skimage.morphology import dilation, skeletonize, square, disk
# from skfuzzy import cmeans
import random
from scipy import stats
from IPython.display import clear_output

warnings.filterwarnings("ignore")


class ClusteringComparison:
    def __init__(self, pca_data, i):
        """Initialize ClusteringComparison with PCA-transformed data."""
        self.i = i
        self.X = pca_data
        self.results_df = pd.DataFrame()
        self.clustering_algorithms = {
            'KMeans': KMeans(),
            'AgglomerativeClustering': AgglomerativeClustering(),
            # 'DBSCAN': DBSCAN(),
            # 'SpectralClustering': SpectralClustering(),
            # 'GaussianMixture': GaussianMixture(),
            'BayesianGaussianMixture': BayesianGaussianMixture(),
            'MeanShift': MeanShift(),
            # 'OPTICS': OPTICS(),
            # 'AffinityPropagation': AffinityPropagation(),
            # 'HDBSCAN': HDBSCAN(),
            # 'Birch': Birch(),
            # 'BisectingKMeans': BisectingKMeans(),
            # 'MiniBatchKMeans': MiniBatchKMeans(),
            'FuzzyCMeans': None,

             }


        self.param_grids = {
            'KMeans': {'n_clusters': [2],
                       'algorithm':['lloyd', 'elkan']
                       },

            'AgglomerativeClustering': {
                'n_clusters': [2],
                'linkage': ['ward', 'complete', 'average', 'single'],
                'metric': ['euclidean', 'manhattan', 'cosine']
                       },

            # 'DBSCAN': {'eps': [0.1, 0.2, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.4, 1.6, 1.8, 2, 2.5, 3, 5, 7, 10, 15, 20],
            #            'min_samples': [1, 2, 3, 5, 10, 20, 30],
            #            'metric': ['euclidean', 'manhattan', 'cosine']
            #            },

            'SpectralClustering': {'n_clusters': [2],
                                   'affinity': ['nearest_neighbors', 'rbf'],
                                  #  'eigen_solver':['arpack', 'lobpcg', 'amg'],
                                   'n_neighbors':[5, 10, 15, 20]
                                   },

            # 'GaussianMixture': {'n_components': [2],
            #                     'covariance_type': ['full', 'tied', 'diag', 'spherical']
            #                     },

            'BayesianGaussianMixture': {'n_components': [2],
                                        'covariance_type': ['full', 'tied', 'diag', 'spherical']
                                        },

            'MeanShift': {'bandwidth': [None, 0.1, 0.2, 0.3, 0.5, 0.7, 1, 1.25, 1.5 , 1.75, 2, 2.5, 3, 3.5, 5, 7]},

            # 'OPTICS': {'min_samples': [0.5, 1.0, 2, 3, 5, 10],
            #            'max_eps': [0.1, 0.2, 0.3, 0.5, 0.7, 1, 3, 5, 7],
            #            'algorithm':['auto'],
            #           #  'p': [1, 2, 4, 8],
            #            'metric': ['euclidean']
            #            },

            'AffinityPropagation': {'damping': [.5, .75, 1.0]},

            # 'HDBSCAN': {'min_cluster_size': [2, 3, 4, 5, 10, 15, 20],
            #             'min_samples':[None, 2, 5, 10, 15, 20],
            #             'cluster_selection_method': ['eom', 'leaf'],
            #             'cluster_selection_epsilon':[0.0, 0.5, 1.0, 2.0, 5.0, 10.0],
            #             'metric': ['euclidean'],
            #             'algorithm': ['auto']
            #             },

            'Birch': {'n_clusters': [2],
                      'branching_factor':[5, 10, 20, 30, 50, 60, 70, 100],
                      'threshold':[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                      },

            # 'BisectingKMeans': {'n_clusters': [2],
            #                     'bisecting_strategy': ['biggest_inertia', 'largest_cluster'],
            #                     },

            # 'MiniBatchKMeans': {'n_clusters': [2],
            #                     'batch_size': [128, 256, 512],
            #                     },

            'FuzzyCMeans': {'c': [2],
                            'm': [1.5, 2.0, 2.5, 3.5],
                            'error': [1e-4, 1e-6, 1e-8],
                            'maxiter': [1e7]
                            }

            }

    def run_clustering(self):
        """Run clustering algorithms and evaluate their performance."""
        results = []

        for algorithm_name, algorithm in self.clustering_algorithms.items():
            grid = ParameterGrid(self.param_grids[algorithm_name])
            for params in grid:

                try:

                    if algorithm_name == 'FuzzyCMeans':
                          cntr, u, _, _, _, _, _ = cmeans(self.X.T, **params)
                          labels = np.argmax(u, axis=0)
                    elif algorithm_name == 'DBSCAN' or algorithm_name == 'HDBSCAN' or algorithm_name == 'OPTICS':
                        alg = algorithm.set_params(**params)
                        labels = alg.fit_predict(self.X)
                        labels = labels + 1
                    else:
                        alg = algorithm.set_params(**params)
                        labels = alg.fit_predict(self.X)


                    n_labels = len(np.unique(labels))

                    # Ensure that the number of labels is sufficient for metric evaluation
                    if 1 < n_labels < len(self.X):
                        silhouette = silhouette_score(self.X, labels)
                        davies_bouldin = davies_bouldin_score(self.X, labels)
                        calinski_harabasz = calinski_harabasz_score(self.X, labels)
                    else:
                        # Assign default values to indicate this configuration was not valid for evaluation
                        silhouette = -999
                        davies_bouldin = 999
                        calinski_harabasz = -999

                except Exception as e:
                    # If fit_predict fails, log the error and continue
                    # print(f"Error with {algorithm_name} and parameters {params}: {e}")
                    silhouette = -999
                    davies_bouldin = 999
                    calinski_harabasz = -999

                results.append({
                    'Algorithm': algorithm_name,
                    'Parameters': str(params),
                    'Silhouette Score': silhouette,
                    'Davies-Bouldin Index': davies_bouldin,
                    'Calinski-Harabasz Index': calinski_harabasz
                })

        self.results_df = pd.DataFrame(results)
        # self.results_df.to_csv(f"results_{self.i}.csv")
        self.best_params_df = self.store_best_parameters(self.results_df)
        # self.best_params_df.to_csv(f"best_params_{self.i}.csv")
        return self.results_df, self.best_params_df


    @staticmethod
    def store_best_parameters(results_df):
        """Find the best parameters for each clustering method."""
        best_params = results_df.groupby('Algorithm')['Silhouette Score'].idxmax()
        return results_df.loc[best_params]

    def plot_results(self):
        """Plot clustering results using different evaluation metrics."""
        metrics = ['Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Index']
        if self.results_df.empty:
            raise ValueError("No results to plot. Please run the clustering first using run_clustering().")

        plt.figure(figsize=(30, 3))
        for i, metric in enumerate(metrics):
            sorted_df = self.results_df.sort_values(by=metric, ascending=(metric != 'Davies-Bouldin Index'))
            plt.subplot(1, len(metrics), i + 1)
            sns.barplot(x=metric, y='Algorithm', hue='Parameters', data=sorted_df, dodge=False)
            plt.title(f'Clustering Algorithms Comparison ({metric})')
            plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

        plt.subplots_adjust(wspace=3.5, hspace=0.3)
        plt.show()


class ExportDatasetoGDrive:
    def __init__(self, input_file, folder_name, start_x, end_x, start_y, end_y):
        """
        Initialize ExportDatasetoGDrive with input file, folder name, and crop coordinates.

        Args:
            input_file (str): Path to the input raster file.
            folder_name (str): Path to the folder where outputs will be saved.
            start_x (int, optional): The starting x pixel coordinate. Default is 0.
            start_y (int, optional): The starting y pixel coordinate. Default is 0.
            width (int, optional): The width of the crop window in pixels. Default is 512.
            height (int, optional): The height of the crop window in pixels. Default is 512.
        """
        self.input_file = input_file
        self.folder_name = folder_name
        self.start_x = start_x
        self.start_y = start_y
        self.end_x = end_x
        self.end_y = end_y
        self.width, self.height = self.end_x-self.start_x, self.end_y-self.start_y

    def save_mask_as_geotiff(self, array, file_name, dtype=gdal.GDT_Byte):
        """
        Save a NumPy array as a GeoTIFF file with real values.

        Args:
            array (numpy.ndarray): The input NumPy array to save.
            file_name (str): Path to the output GeoTIFF file.
            dtype (gdal.DataType, optional): The GDAL data type for the output file. Default is gdal.GDT_UInt16.
        """
        output_file = os.path.join(self.folder_name, file_name.lstrip('/'))
        dataset = gdal.Open(self.input_file, gdalconst.GA_ReadOnly)

        if not dataset:
            raise FileNotFoundError(f"Unable to open file: {self.input_file}")

        # array = dataset.ReadAsArray()
        transform = dataset.GetGeoTransform()
        projection = dataset.GetProjection()



        array = array[self.start_y : self.end_y,
                      self.start_x : self.end_x]

        driver = gdal.GetDriverByName('GTiff')
        out_dataset = driver.Create(output_file, self.width, self.height, 1, dtype)

        if not out_dataset:
            raise RuntimeError(f"Unable to create output file: {output_file}")

        out_dataset.SetGeoTransform(transform)
        out_dataset.SetProjection(projection)
        out_dataset.GetRasterBand(1).WriteArray(array)

        out_dataset.FlushCache()
        out_dataset = None  # Close the dataset



    def crop_raster_as_geotiff(self, file_name, selected_bands=None, dtype = gdal.GDT_UInt32):
        """
        Crop a raster file based on pixel coordinates and save it as a new GeoTIFF, selecting specific bands.

        Args:
            file_name (str): Path to the output GeoTIFF file.
            selected_bands (list of int, optional): List of band indices to export. If None, export all bands.
        """
        output_file = os.path.join(self.folder_name, file_name.lstrip('/'))

        dataset = gdal.Open(self.input_file, gdalconst.GA_ReadOnly)

        if not dataset:
            raise FileNotFoundError(f"Unable to open file: {self.input_file}")

        if selected_bands is None:
            selected_bands = list(range(1, dataset.RasterCount + 1))

        if max(selected_bands) > dataset.RasterCount or min(selected_bands) < 1:
            raise ValueError("Selected bands are out of range.")

        driver = gdal.GetDriverByName('GTiff')
        out_dataset = driver.Create(output_file, self.width, self.height, len(selected_bands), dtype)

        if not out_dataset:
            raise RuntimeError(f"Unable to create output file: {output_file}")

        transform = dataset.GetGeoTransform()
        new_transform = (
            transform[0] + self.start_x * transform[1] + self.start_y * transform[2],
            transform[1],
            transform[2],
            transform[3] + self.start_x * transform[4] + self.start_y * transform[5],
            transform[4],
            transform[5]
        )

        out_dataset.SetGeoTransform(new_transform)
        out_dataset.SetProjection(dataset.GetProjection())

        for i, band_index in enumerate(selected_bands, start=1):
            band = dataset.GetRasterBand(band_index)
            data = band.ReadAsArray(self.start_x, self.start_y, self.width, self.height)*10000
            out_band = out_dataset.GetRasterBand(i)
            out_band.WriteArray(data)

        out_dataset.FlushCache()
        dataset = None  # Close the original dataset
        out_dataset = None  # Close the output dataset

class ImageClustering:
    """

    Attributes:
      clustering_methods:
      best_parameters:
      X:
      labels:
      masks:
    """
    def __init__(self, clustering_methods, best_parameters, pca_data):
        """Initialize ImageClustering with clustering methods, best parameters, and PCA data."""
        self.clustering_methods = clustering_methods
        self.best_parameters = best_parameters
        self.X = pca_data
        self.labels = []
        self.masks = []

    def clustering(self):
        """Apply the best clustering methods to the PCA data."""
        self.labels = []
        for algorithm_name, algorithm in self.clustering_methods.items():
            params = eval(self.best_parameters[self.best_parameters['Algorithm'] == algorithm_name]['Parameters'].values[0])
            if algorithm_name == 'FuzzyCMeans':
              cntr, u, _, _, _, _, _ = cmeans(self.X.T, **params)
              self.labels.append(np.argmax(u, axis=0))
            elif algorithm_name == 'DBSCAN' or algorithm_name == 'HDBSCAN' or algorithm_name == 'OPTICS':
              alg = algorithm.set_params(**params)
              self.labels.append(alg.fit_predict(self.X)+1)
            else:
              alg = algorithm.set_params(**params)
              self.labels.append(alg.fit_predict(self.X))
        return self.labels

    def create_mask_from_clustering(self, label_image, NDWI):
        """Create a mask from clustering labels."""
        self.water_labeled = []
        unique_segments = np.unique(label_image)

        for cluster_membership in self.labels:
            mask_image = np.zeros_like(label_image)

            for i, segment in enumerate(unique_segments):
                # Find which cluster this segment belongs to and assign the corresponding label
                mask_image[label_image == segment] = cluster_membership[i]

            self.water_labeled.append(self.labeling(mask_image, NDWI))
            # masks.append(mask_image)

        # Return the final mask as a boolean array
        # water_labeled = self.labeling(masks, NDWI)
        self.water_labeled.append(self.Uncertainty(self.water_labeled))  # Assuming you want to add the uncertainty mask to the list
        return self.water_labeled


    def labeling(self, mask, NDWI):
        # Water_labeled = []
        unique = np.unique(mask)
        water_label = np.zeros_like(mask)

        if len(unique) > 1:
            values = [np.nanmean(NDWI[mask == value]) for value in unique]
            # print(f"Unique values: {unique}, NDWI values: {values}")
            max_value_index = np.argmax(values)
            # print(f"Labeling segment with value: {unique[max_value_index]}")
            water_label[mask == unique[max_value_index]] = 1
        else:
            # print("Only one unique value in mask. Creating empty water label.")
            water_label = np.zeros_like(mask)

        # Water_labeled.append(water_label)

        return water_label


    def Uncertainty(self, masks):
         unc = stats.mode(np.asarray(masks), axis=0, keepdims=True).mode
        #  print(unc[0, :, :].shape)
         return unc[0, :, :]


    def plot_images(self, index, image_array, label_image, masks):
        """Plot images including MNDWI, water boundary, label image, and clustering masks."""
        dim, row, col = image_array.shape
        c_x , c_y = int(row/2), int(col/2)

        xstart, xend = c_x - 256, c_x + 256
        ystart, yend =  c_y - 256, c_y + 256

        fig, axes = plt.subplots(1, 4 + len(masks), figsize=(50, 5))
        fig.suptitle(f'Patch: {index + 1}')
        titles = ['MNDWI', 'VV', 'Water Boundary', 'Label Image', 'Threshold'] + list(self.clustering_methods.keys()) + ['Uncertainty']

        images = [image_array[-5, xstart:xend, ystart:yend],image_array[-5, xstart:xend, ystart:yend], image_array[-1, xstart:xend, ystart:yend], label_image[xstart:xend, ystart:yend]] + [mask[xstart:xend, ystart:yend] for mask in masks]

        cmaps = ['Spectral','gray' ,'gray', 'jet'] + ['Blues'] * len(masks)

        for ax, img, title, cmap in zip(axes, images, titles, cmaps):
            ax.imshow(img, cmap=cmap)
            ax.set_title(title)
            ax.axis('off')
        plt.subplots_adjust(wspace=.5, hspace=0.5)  # Adjust wspace and hspace as needed
        plt.show()




class ImageProcessor:
    def __init__(self, path_file, index_band, root_file, thre_mean=0.0, thre_var=0.03,
                 metadata = None, path_metadata = None):
        """
        Initialize the ImageProcessor with paths and parameters.

        Args:
            path_file (str): Path to the folder containing image files.
            index_band (list): List of band indices to be used for feature extraction.
            root_file (str): Root file name or path for export purposes.
            thre_mean (float, optional): Threshold for mean value in mask creation. Default is 0.0.
            thre_var (float, optional): Threshold for variance in mask creation. Default is 0.03.
            metadata (dict, optional): Metadata for export purposes. Default is None.
            path_metadata (str, optional): Path to the metadata file. Default is None.



        """
        self.path_file = path_file
        self.thre_mean = thre_mean
        self.thre_var = thre_var
        self.index_band = index_band
        self.file_names = sorted(os.listdir(path_file), key=lambda x: int(x.split('_')[-1].split('.')[0]))
        self.metadata = metadata
        self.root_file = root_file
        self.root_folder = root_file

    def Data_infromation(self):

        self.path_metadata = os.path.join(self.root_folder, "data_information.csv")
        if not os.path.exists(self.path_metadata):
           if self.metadata is not None:
               self.data_infromation = pd.DataFrame(columns=self.metadata.keys())
               self.data_infromation.to_csv(self.path_metadata)

           else:
               self.data_infromation = pd.DataFrame(columns=['id'])
               self.data_infromation.to_csv(self.path_metadata)
        else:
            self.data_infromation = pd.read_csv(self.path_metadata)


        return self.data_infromation




    def check_and_create_folders(self, folders=["S1", "S2", "gt"]):
        for folder in folders:
            folder_path = os.path.join(self.root_file, folder)
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)

        return folder_path

    def id_creator(self):
            # print(len(self.data_infromation))
            self.id = len(self.data_infromation) + 1
            self.id = str(self.id ).zfill(5)
            # print(self.id)

            return self.id

    def update_information(self):

        if self.metadata is not None:
            new_row = {'Climate Zone': self.metadata.get('Climate Zone'),
                      'Agro-Ecological Zone': self.metadata.get('Agro-Ecological Zone'),
                      'Country Name': self.metadata.get('Country Name'),
                      'Region Name': self.metadata.get('Region Name'),
                      'id': str(self.id),
                      'S1': 'Yes',
                      'S2': 'Yes',
                      'gt': 'Yes',
                      'Quality Check': None,
                      'Date': None
                      }

        else:
            new_row = {'id': str(self.id)}
        self.data_infromation = pd.concat([self.data_infromation, pd.DataFrame([new_row])], ignore_index=True)

        return self.data_infromation



    @staticmethod
    def create_mask_threshold(array_patch, label_image, thre_mean):
        """
        Create a binary mask based on a threshold.

        Args:
            array_patch (np.ndarray): The image array patch.
            label_image (np.ndarray): The labeled image to base the mask on.
            thre_mean (float): The threshold mean value for mask creation.

        Returns:
            np.ndarray: Binary mask image.
        """
        mask_image = np.zeros_like(label_image)
        for segment in np.unique(label_image):
            pattern = label_image == segment
            if np.mean(array_patch[pattern]) > thre_mean:
                mask_image[pattern] = 1
        return mask_image

    @staticmethod
    def create_label_image(array):
        """
        Create a label image for the given array by dilating and labeling the array.

        Args:
            array (np.ndarray): The input array to be labeled.

        Returns:
            np.ndarray: Labeled image.
        """
        dilated_image = dilation(array, disk(1))  # Dilate to connect disjoint lines
        return label(~array.astype(bool), connectivity=1)

    def create_feature_matrix(self, image_array, label_image):
        """
        Create a feature matrix from the image array and label image.

        Args:
            image_array (np.ndarray): The image array.
            label_image (np.ndarray): The labeled image.

        Returns:
            np.ndarray: Feature matrix.
        """
        feature_matrix = image_array[self.index_band, :, :]
        n_segments = len(np.unique(label_image))
        n_features = len(self.index_band) * 3
        features = np.zeros((n_segments, n_features))

        for i, segment in enumerate(np.unique(label_image)):
            pattern = label_image == segment
            features[i, :len(self.index_band)] = np.nanmedian(feature_matrix[:, pattern], axis=1)
            features[i, len(self.index_band):2 * len(self.index_band)] = np.nanvar(feature_matrix[:, pattern], axis=1)
            features[i, 2 * len(self.index_band):] = np.nanmean(feature_matrix[:, pattern], axis=1)

        features = np.nan_to_num(features, nan=0)
        return features, n_segments


    @staticmethod
    def normalize_features(features):
        """
        Normalize the feature matrix.

        Args:
            features (np.ndarray): The feature matrix to be normalized.

        Returns:
            np.ndarray: Normalized feature matrix.
        """
        return MinMaxScaler().fit_transform(features)

    @staticmethod
    def pca_features(features, n_components=5):
        """
        Perform PCA on the feature matrix.

        Args:
            features (np.ndarray): The feature matrix.
            n_components (int, optional): Number of principal components. Default is 5.

        Returns:
            np.ndarray: Transformed feature matrix after PCA.
        """
        n_components = min(n_components, features.shape[0], features.shape[1])
        pca = PCA(n_components=n_components)
        return pca.fit_transform(features)

    def plot_images_polygon(self, index, image_array):
        """Plot images including MNDWI, water boundary, label image, and clustering masks."""
        dim, row, col = image_array.shape
        c_x , c_y = int(row/2), int(col/2)

        xstart, xend = c_x - 256, c_x + 256
        ystart, yend =  c_y - 256, c_y + 256

        fig, axes = plt.subplots(1, 3, figsize=(10, 5))
        fig.suptitle(f'Patch: {index + 1}')
        titles = ['RGB', 'VV', 'Mask']

        rgb = 5.*(image_array[[3, 2, 1]].transpose(1, 2, 0)) / 10000

        images = [rgb[xstart:xend, ystart:yend, :], image_array[-2, xstart:xend, ystart:yend], image_array[-1, xstart:xend, ystart:yend]]

        cmaps = [None,'gray', 'Blues']

        for ax, img, title, cmap in zip(axes, images, titles, cmaps):
            ax.imshow(img, cmap=cmap)
            ax.set_title(title)
            ax.axis('off')
        plt.subplots_adjust(wspace=.5, hspace=0.5)  # Adjust wspace and hspace as needed
        plt.show()

    def process_images(self):
        """
        Process images to apply clustering and visualize results.
        """
        self.root_file = self.check_and_create_folders(folders=[self.metadata.get('Region Name')])
        self.check_and_create_folders(folders=["S1", "S2", "gt"])

        self.data_information = self.Data_infromation()

        for i, file_name in enumerate(self.file_names):
            file_path = os.path.join(self.path_file, file_name)
            image = gdal.Open(file_path, gdalconst.GA_ReadOnly)

            if image is None:
                print(f"Failed to open image file: {file_path}. Skipping this file.")
                pass

            image_array = image.ReadAsArray()

            if isinstance(image_array, np.ndarray) and image_array.ndim == 3 and image_array.shape[0] == 16:

              dim, row, col = image_array.shape
              c_x , c_y = int(row/2), int(col/2)

              xstart, xend = c_x - 256, c_x + 256
              ystart, yend =  c_y - 256, c_y + 256

              """
              label_image = self.create_label_image(image_array[-1, :, :])
              mask_threshold = self.create_mask_threshold(image_array[-5, :, :], label_image, self.thre_mean)

              features, patterns = self.create_feature_matrix(image_array, label_image)
              features_transformed = self.normalize_features(features)
              pca_result = self.pca_features(features_transformed)

              # Assuming ClusteringComparison and ImageClustering classes are implemented elsewhere
              clustering_comparison = ClusteringComparison(pca_result, i)
              results, best_params = clustering_comparison.run_clustering()

              image_clustering = ImageClustering(clustering_comparison.clustering_algorithms, best_params, pca_result)
              image_clustering.clustering()
              masks = image_clustering.create_mask_from_clustering(label_image, image_array[-5, :, :])
              masks = [mask_threshold] + (masks)

              """
              self.id = self.id_creator()

              export = ExportDatasetoGDrive(input_file=file_path, folder_name=self.root_file, start_x=xstart, end_x=xend, start_y=ystart, end_y=yend)

              export.crop_raster_as_geotiff(file_name=f'S2/S2_{self.id}.tif', selected_bands=list(range(1, 14)))
              export.crop_raster_as_geotiff(file_name=f'S1/S1_{self.id}.tif', selected_bands=[14, 15], dtype=gdal.GDT_Int32)
              export.crop_raster_as_geotiff(file_name = f'/gt/gt_{self.id}.tif', selected_bands=[16], dtype=gdal.GDT_Byte)

              # export.save_mask_as_geotiff(file_name = f'/gt/gt_{self.id}.tif', array = masks[-1])
              self.data_information = self.update_information()

              self.plot_images_polygon(i, image_array)

            else:
              pass
        self.data_information.to_csv(os.path.join(self.root_folder, 'data_information.csv'))


if __name__ == "__main__":
    path_file = "/content/drive/MyDrive/Patch_Huron"
    index_band = [13, 14, 15, 16, 17, 18, 19]
    processor = ImageProcessor(path_file=path_file, index_band=index_band, root_file='/content/drive/MyDrive/Patch_Dataset/',
                               metadata= {'Climate Zone': 'Temperate',
                                          'Agro-Ecological Zone': 'Moist',
                                          'Country Name': 'Canada',
                                          'Region Name': 'Huron Lake',
                                          'id': None,
                                          'S1': None,
                                          'S2': None,
                                          'gt': None,
                                          'Quality Check': None,
                                          'Date': None
                                          })

    processor.process_images()
